今天记录一个agent简单实现案例来理解学习下相关内容

OpenRouter 是聚合各类大模型的平台，SDK 封装了其底层 HTTP API，让你不用手动写网络请求

In [5]:
# 导入 OpenAI SDK（兼容 OpenRouter）
from openai import OpenAI
import os

可以把SDK（Software Development Kit，软件开发工具包） 形式的 API 接口，理解为 “封装好的工具箱”：
SDK 形式的 API 是 “带说明书 + 现成工具的工具箱”，开发者不用关心底层的网络请求、数据格式转换，直接调用工具箱里的 “函数 / 方法” 就能实现功能。

In [18]:
# ========== 关键修改：手动填入你的 API Key ==========
# 替换成你从 OpenRouter 官网获取的 Key（格式：sk-xxxxxxxxxxxxxxxxxxxx）
# 安全原因考虑这里会删去本人API-key,已示例示范
API_KEY = "格式：sk-xxxxxxxxxxxxxxxxxxxx"

# 初始化客户端
with OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY  # 直接传入手动填写的 Key
)as client:
    
    
# 引入变量messages记录历史对话内容
    messages = [{"role": "assistant", "content": """
        你的目标是完成用户的任务，你必须选择下面的其中一种格式进行回复:
        1。如果你认为需要执行命令，则输出'命令:XXX'，XXX为命令本身，不要用任何的格式，不要解释
        2.如果你认为不需要执行命令，则输出'完成:XXX'，XXX为你的总结信息
    """}]

    while True:
        user_input = input("\nyour: ")
        messages.append({"role": "user", "content": user_input})
        print("\n------agent 循环开始------")
        while True:
            # 由于没有token这里调用免费模型
            response = client.chat.completions.create(
                model="nvidia/nemotron-3-super-120b-a12b:free",
                messages=messages
            )
            reply = response.choices[0].message.content
            messages.append({"role": "assistant", "content":reply})
            # 打印回复
            print(f"[AI]{reply}")
            if reply.strip().startswith("完成:"):
                print("\n------agent 循环结束------")
                print(f"[AI]{reply.strip().split('完成:')[1].strip()}")
                break

            command = reply.strip().split("命令:")[1].strip()
            command_result = os.popen(command).read()
            # 这里笔误吧content写成contest才导致后面报错现已修复
            content = f"执行完毕 {command_result}"
            print(f" [agent]{content}")
            messages.append({"role": "user", "content": content})



your:  在Linux系统中生成一份名为text的文本文件，内容为牢蒋牛福



------agent 循环开始------
[AI]命令:echo "牢蒋牛福" > text


NameError: name 'content' is not defined

遇到报错：
```
AttributeError                            Traceback (most recent call last)
Cell In[9], line 18
     16 messages=[{"role": "user", "content": user_input}]
     17 # 由于没有token这里调用免费模型
---> 18 response = client.chat.send(
     19     model="qwen/qwen-2-72b-instruct:free",
     20     messages=messages
     21 )
     22 reply = response.choices[0].message.content
     23 messages.append({"role": "assistant", "content":reply})

AttributeError: 'Chat' object has no attribute 'send'
```
```
BadRequestError                           Traceback (most recent call last)
Cell In[16], line 26
     23 print("\n------agent 循环开始------")
     24 while True:
     25     # 由于没有token这里调用免费模型
---> 26     response = client.chat.completions.create(
     27         model="step-ai/step-3-5-flash:free",
     28         messages=messages
     29     )
     30     reply = response.choices[0].message.content
     31     messages.append({"role": "assistant", "content":reply})
```

由于时间原因，后面在做补充